In [56]:
import pandas as pd
import numpy as np

In [57]:
df = pd.read_csv("../northbeam_feature_engineered.csv")

In [58]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
contract_value,424.0,91926.470472,215886.949785,-4500.000000,3544.005000,9726.250000,72607.442500,1322115.99
seats_licensed,424.0,201.580189,327.647670,5.000000,25.000000,68.000000,200.250000,1488.00
seats_active,424.0,134.674528,242.465289,2.000000,16.000000,39.500000,124.000000,1301.00
mau_m5,424.0,82.992925,169.800766,0.000000,5.000000,18.500000,74.250000,1206.00
mau_m4,424.0,81.372642,164.123873,0.000000,5.000000,18.000000,69.250000,1151.00
mau_m3,424.0,77.608491,153.878978,0.000000,5.000000,19.000000,72.000000,1147.00
mau_m2,424.0,71.823113,150.319060,0.000000,3.000000,16.000000,68.250000,1080.00
mau_m1,424.0,69.214623,148.807915,0.000000,3.000000,15.500000,70.000000,1069.00
mau_m0,424.0,65.181604,153.742770,0.000000,1.000000,13.000000,60.250000,1154.00
support_tickets_90d,424.0,4.573113,2.629948,0.000000,3.000000,4.000000,6.000000,13.00


In [59]:
df["seat_utilization"] = df["seat_utilization"].clip(upper=100)

In [60]:
df["seat_utilization"].describe()

count    424.000000
mean      66.372767
std       18.325938
min       33.333333
25%       50.000000
50%       67.152322
75%       81.947439
max      100.000000
Name: seat_utilization, dtype: float64

In [61]:
# Thresholds: 82 ≈ p75, 67 ≈ p50 (median), 50 ≈ p25
def seat_score(utilization):

    if utilization >= 82:  # top quartile
        return 25

    elif utilization >= 67:  # above median
        return 20

    elif utilization >= 50:  # above bottom quartile
        return 10

    else:
        return 0       # bottom quartile — low adoption risk

In [62]:
df["seat_score"] = df["seat_utilization"].apply(seat_score)

df[["seat_utilization","seat_score"]].head()

,seat_utilization,seat_score
0,39.534884,0
1,74.305556,20
2,46.153846,0
3,75.000000,20
4,100.000000,25


In [63]:
# Create flag for missing CSAT values
df["csat_missing"] = df["csat"].isnull().astype(int)

# Fill missing values with median
median_csat = df["csat"].median()

df["csat"] = df["csat"].fillna(median_csat)

In [64]:
df["csat"].isnull().sum()

np.int64(0)

In [65]:
# Thresholds: 4.2 = p75, 3.7 = p50, 3.1 = p25
def csat_score(csat):

    if csat >= 4.2:  # top quartile
        return 20

    elif csat >= 3.7: # above median
        return 15

    elif csat >= 3.1:  # above bottom quartile
        return 10

    else:           # bottom quartile
        return 0

In [66]:
df["csat_score"] = df["csat"].apply(csat_score)

df[["csat","csat_score"]].head()

,csat,csat_score
0,3.7,15
1,4.3,20
2,3.7,15
3,3.1,10
4,3.7,15


In [67]:
# Thresholds: 68 ≈ p75, 17 ≈ p50, 5 ≈ p25
def mau_score(avg_mau):

    if avg_mau >= 68:  # top quartile
        return 20

    elif avg_mau >= 17: # above median
        return 15

    elif avg_mau >= 5:  # above bottom quartile
        return 10

    else:    # bottom quartile
        return 0

In [68]:
df["mau_score"] = df["avg_mau"].apply(mau_score)

df[["avg_mau","mau_score"]].head()

,avg_mau,mau_score
0,0.833333,0
1,4.500000,0
2,10.500000,10
3,37.833333,15
4,5.166667,10


In [69]:
# Thresholds: 3 ≈ p75, 0 ≈ p50 (median), -10 ≈ p25
def usage_score(trend):

    if trend > 3:  # top quartile
        return 15

    elif trend >= 0: # above median
        return 10

    elif trend >= -10:  # above bottom quartile
        return 5

    else:  #bottom quartile
        return 0

In [70]:
df["usage_score"] = df["usage_trend"].apply(usage_score)

df[["usage_trend","usage_score"]].head()

,usage_trend,usage_score
0,1,10
1,2,10
2,0,10
3,-48,0
4,-7,5


In [71]:
def support_score(tickets):

    if tickets <= 3:
        return 10

    elif tickets <= 6:
        return 7

    elif tickets <= 9:
        return 4

    else:
        return 0

In [72]:
df["support_score"] = df["support_tickets_90d"].apply(support_score)

df[["support_tickets_90d","support_score"]].head()

,support_tickets_90d,support_score
0,6,7
1,0,10
2,6,7
3,7,4
4,1,10


In [73]:
# Escalation count: 0 = no issue, 1 = minor, 2 = concerning, 3+ = serious pattern
def escalation_score(escalations):

    if escalations == 0:
        return 10

    elif escalations == 1:
        return 7

    elif escalations == 2:
        return 4

    else:
        return 0

In [74]:
df["escalation_score"] = df["escalations_90d"].apply(escalation_score)

df[["escalations_90d","escalation_score"]].head()

,escalations_90d,escalation_score
0,0,10
1,0,10
2,1,7
3,0,10
4,0,10


In [75]:
def revenue_score(contract):

    if contract >= 72607:
        return 20

    elif contract >= 9726:
        return 15

    elif contract >= 3544:
        return 10

    else:
        return 5

In [76]:
df["revenue_score"] = df["annual_contract_value"].apply(revenue_score)

df[["annual_contract_value","revenue_score"]].head()

,annual_contract_value,revenue_score
0,1660.09,5
1,67135.04,15
2,5723.84,10
3,61588.39,15
4,2765.58,5


In [77]:
def renewal_score(days):

    if days <= 30:
        return 20

    elif days <= 60:
        return 15

    elif days <= 90:
        return 10

    else:
        return 5

In [78]:
df["renewal_score"] = df["days_to_renewal"].apply(renewal_score)
df[["days_to_renewal","renewal_score"]].head()

,days_to_renewal,renewal_score
0,207.0,5
1,186.0,5
2,210.0,5
3,387.0,5
4,18.0,20


In [79]:
df["customer_health_score"] = (
      df["seat_score"]
    + df["mau_score"]
    + df["csat_score"]
    + df["usage_score"]
    + df["support_score"]
    + df["escalation_score"]
    + df["revenue_score"]
    + df["renewal_score"] 
     
)

In [80]:
df[[
    "seat_score",
    "mau_score",
    "csat_score",
    "usage_score",
    "support_score",
    "escalation_score",
    "revenue_score",
    "customer_health_score",
    "renewal_score"
    
]].head()

,seat_score,mau_score,csat_score,usage_score,support_score,escalation_score,revenue_score,customer_health_score,renewal_score
0,0,0,15,10,7,10,5,52,5
1,20,0,20,10,10,10,15,90,5
2,0,10,15,10,7,7,10,64,5
3,20,15,10,0,4,10,15,79,5
4,25,10,15,5,10,10,5,100,20


In [81]:
df["customer_health_score"].describe()

count    424.000000
mean      82.742925
std       17.755163
min       34.000000
25%       70.000000
50%       82.000000
75%       95.000000
max      125.000000
Name: customer_health_score, dtype: float64

In [82]:
def customer_status(score):

    if score >= 95:
        return "Healthy"

    elif score >= 70:
        return "Monitor"

    else:
        return "At Risk"

In [83]:
df["customer_status"] = df["customer_health_score"].apply(customer_status)

df[["customer_health_score","customer_status"]].head()

,customer_health_score,customer_status
0,52,At Risk
1,90,Monitor
2,64,At Risk
3,79,Monitor
4,100,Healthy


In [84]:
# Create final business priority score
# Higher score = higher intervention priority

max_score = df["customer_health_score"].max()

df["priority_score"] = (
      (max_score - df["customer_health_score"]) * 0.5
    + df["revenue_score"] * 0.3
    + df["renewal_score"] * 0.2
)


# Sort by highest priority score

df = df.sort_values(
    by="priority_score",
    ascending=False
)


# Assign rank

df["priority_rank"] = range(1, len(df)+1)

In [85]:
top_40 = df.head(40)

In [86]:
top_40

,account_id,account_name,segment,industry,region,signup_date,contract_start_date,contract_end_date,billing_term,contract_value,...,mau_score,usage_score,support_score,escalation_score,revenue_score,renewal_score,customer_health_score,customer_status,priority_score,priority_rank
186,ACC-1091,Yarrow Group,SMB,Manufacturing,EMEA,2023-05-24,2023-05-24,2026-08-19,monthly,476.33,...,0,5,7,7,5,10,34,At Risk,49.0,1
244,ACC-1027,Helix Partners,SMB,Construction,North America,2024-11-19,2024-11-19,2026-08-05,monthly,614.96,...,0,10,10,4,5,10,39,At Risk,46.5,2
192,ACC-1344,Pinnacle Media,SMB,SaaS,North America,2025-11-06,2025-11-06,2026-11-10,annual,4769.00,...,0,5,10,10,10,5,40,At Risk,46.5,3
307,ACC-1169,Vista Partners,Mid-Market,Education,North America,2023-01-07,2023-01-07,2027-01-03,annual,73368.19,...,15,0,7,0,20,5,47,At Risk,46.0,4
97,ACC-1157,Quantum Supply,SMB,Healthcare,APAC,2022-11-11,2022-11-11,2026-08-28,monthly,362.79,...,0,5,7,4,5,10,41,At Risk,45.5,5
411,ACC-1345,Garnet Group,SMB,Hospitality,EMEA,2026-04-20,2026-04-20,2026-12-09,annual,4536.69,...,0,5,7,7,10,5,44,At Risk,44.5,6
206,ACC-1297,Maple Dynamics,Mid-Market,FinServ,North America,2025-02-23,2025-02-23,2026-10-06,annual,49384.31,...,0,5,4,4,15,5,48,At Risk,44.0,7
380,ACC-1195,Juniper Foods,SMB,Hospitality,APAC,2024-11-22,2024-11-22,2026-07-27,annual,2389.31,...,0,5,4,7,5,15,46,At Risk,44.0,8
17,ACC-1235,Larkfield Foods,Mid-Market,FinServ,EMEA,2026-03-21,2026-03-21,2026-04-04,monthly,3326.91,...,10,0,7,7,5,20,49,At Risk,43.5,9
232,ACC-1143,Pinnacle Capital,SMB,Hospitality,North America,2023-07-04,2023-07-04,2026-10-23,monthly,319.48,...,0,5,7,7,5,5,44,At Risk,43.0,10


In [87]:
# Complete dataset with health scores
df.to_csv("../northbeam_customer_health_scores.csv", index=False)

top_40.to_csv("../top_40_priority_customers.csv", index=False)

In [55]:
df.columns.tolist()

['account_id',
 'account_name',
 'segment',
 'industry',
 'region',
 'signup_date',
 'contract_start_date',
 'contract_end_date',
 'billing_term',
 'contract_value',
 'seats_licensed',
 'seats_active',
 'mau_m5',
 'mau_m4',
 'mau_m3',
 'mau_m2',
 'mau_m1',
 'mau_m0',
 'support_tickets_90d',
 'escalations_90d',
 'last_login_date',
 'csat',
 'data_quality_flag',
 'customer_tenure_days',
 'seat_utilization',
 'annual_contract_value',
 'avg_mau',
 'usage_trend',
 'days_to_renewal',
 'high_support',
 'has_escalation',
 'seat_score',
 'csat_missing',
 'csat_score',
 'mau_score',
 'usage_score',
 'support_score',
 'escalation_score',
 'customer_health_score',
 'customer_status']